# Week 2 — Feature Engineering

**Input:** `data/parsed_replays.csv` — 194 games, one row per game, nested dicts as strings

**Output:** `data/features.csv` — one row per player per game (2 rows per 1v1), flat columns, model-ready

**Steps:**
1. Load and profile the raw CSV — nulls, distributions, outliers
2. Parse nested string columns back to Python dicts
3. Flatten to per-player rows
4. Add delta features (your time vs opponent's time)
5. Elo bucketing
6. Tag MY_PROFILE_ID rows

## Setup

In [ ]:
import ast
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)

RAW_PATH      = '../data/parsed_replays.csv'
OUTPUT_PATH   = '../data/features.csv'
MY_PROFILE_ID = 3134896  # TheRealRuClEsHe — confirmed from Feb 2026 replays

print('Imports OK')

## Section 1 — Load and Profile Raw Data

Load the CSV and inspect it before touching anything. Document observations below.

In [2]:
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape}  ({df.shape[0]} games, {df.shape[1]} columns)')
print()
print('Columns:', list(df.columns))
df.head(3)

Shape: (194, 8)  (194 games, 8 columns)

Columns: ['filepath', 'duration_min', 'resign_player', 'resign_time_min', 'age_up_times', 'leaderboard_ratings', 'de_players', 'error']


,filepath,duration_min,resign_player,resign_time_min,age_up_times,leaderboard_ratings,de_players,error
0,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\AgeIIDE_...,22.06,2.0,22.06,"{1: {'feudal': 6.68, 'castle': 20.16}, 2: {'feudal': 6.73}}","{0: 1743, 1: 1678}","{1: {'number': 1, 'color_id': 1, 'team_id': 2, 'ai_name': b'', 'name': b'kat...",NaN
1,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\AgeIIDE_...,27.02,1.0,27.02,"{2: {'feudal': 6.58, 'castle': 19.25}, 1: {'feudal': 7.41, 'castle': 19.48}}","{1: 1715, 0: 1665}","{1: {'number': 1, 'color_id': 1, 'team_id': 2, 'ai_name': b'', 'name': b'Bro...",NaN
2,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\AgeIIDE_...,19.02,1.0,19.02,"{1: {'feudal': 6.75}, 2: {'feudal': 7.08, 'castle': 18.63}}","{1: 1578, 0: 1572}","{1: {'number': 1, 'color_id': 1, 'team_id': 2, 'ai_name': b'', 'name': b'qdw...",NaN


In [ ]:
import matplotlib.pyplot as plt

# Text summary — all columns
print('=== Null counts (all columns) ===')
print(df.isnull().sum())
print()
print('=== Dtypes ===')
print(df.dtypes)

# Bar chart — only columns with nulls, excluding 'error' (null = no parse errors = good)
null_counts = df.isnull().sum().drop('error', errors='ignore')
null_counts = null_counts[null_counts > 0]

if len(null_counts) > 0:
    null_counts.plot(kind='bar', edgecolor='black', color='steelblue')
    plt.title('Null counts per column (excluding error — null there means no parse failures)')
    plt.ylabel('Null count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No nulls in any column (excluding error).')

In [ ]:
print('=== Numeric column distributions ===')
print(df[['duration_min']].describe())

In [6]:
# Preview the nested string columns before parsing
print('=== age_up_times sample (raw string) ===')
for i, val in enumerate(df['age_up_times'].head(5)):
    print(f'  [{i}] {val}')

print()
print('=== leaderboard_ratings sample (raw string) ===')
for i, val in enumerate(df['leaderboard_ratings'].head(5)):
    print(f'  [{i}] {val}')

=== age_up_times sample (raw string) ===
  [0] {1: {'feudal': 6.68, 'castle': 20.16}, 2: {'feudal': 6.73}}
  [1] {2: {'feudal': 6.58, 'castle': 19.25}, 1: {'feudal': 7.41, 'castle': 19.48}}
  [2] {1: {'feudal': 6.75}, 2: {'feudal': 7.08, 'castle': 18.63}}
  [3] {2: {'feudal': 6.26}, 1: {'feudal': 6.61}}
  [4] {1: {'feudal': 6.22, 'castle': 19.51}, 2: {'feudal': 7.08, 'castle': 17.85}}

=== leaderboard_ratings sample (raw string) ===
  [0] {0: 1743, 1: 1678}
  [1] {1: 1715, 0: 1665}
  [2] {1: 1578, 0: 1572}
  [3] {0: 2224, 1: 2348}
  [4] {1: 1745, 0: 1705}


## Section 1 Observations

*(Fill in after running the cells above)*

**Nulls:**
- `resign_player` / `resign_time_min`: expected nulls = games ending by disconnect or timeout (no RESIGN action)
- `error` column: all NaN — safe to drop

**Duration distribution:**
- *(note min, max, median after running)*
- Any suspiciously short games that passed the 8-min filter?

**Nested columns:**
- `age_up_times`: dict keyed by player_num → {age: minutes}. Some players may be missing castle/imperial if game ended early
- `leaderboard_ratings`: dict keyed by leaderboard number (0/1, not always matching player_num 1/2 — known mismatch)
- `de_players`: contains profile_id needed for MY_PROFILE_ID tagging

**Missing columns (were used as filters, not saved):**
- `map_id`, `rated`, `num_players` — all games are Arabia/rated/1v1 by definition, will add back as constants

## Section 2 — Clean and Flatten to Per-Player Rows

**What this section does:**
1. Drop `error` column (all NaN)
2. Drop 19 rows with null `resign_player` — no winner recorded, unusable as training data (DEC-002)
3. Parse nested string columns back to Python dicts using `ast.literal_eval`
4. Expand each game into 2 player rows — one per player
5. Flatten `age_up_times` and `leaderboard_ratings` into proper columns
6. Tag `is_me` using `MY_PROFILE_ID` (DEC-004)
7. Add back constant columns `map_id`, `rated`, `num_players` (DEC-005)

In [7]:
# ── Step 1 & 2: Drop error column and null resign rows ────────────────────────
df = df.drop(columns=['error'])
before = len(df)
df = df.dropna(subset=['resign_player'])
print(f'Dropped {before - len(df)} rows with no resign recorded — {len(df)} games remain')

# ── Step 3: Parse nested string columns back to Python dicts ──────────────────
# CSV stores dicts as strings — ast.literal_eval converts them back safely
df['age_up_times']        = df['age_up_times'].apply(ast.literal_eval)
df['leaderboard_ratings'] = df['leaderboard_ratings'].apply(ast.literal_eval)
df['de_players']          = df['de_players'].apply(ast.literal_eval)
print('Nested columns parsed OK')

Dropped 19 rows with no resign recorded — 175 games remain
Nested columns parsed OK


In [8]:
# ── Steps 4–7: Expand to per-player rows ──────────────────────────────────────
# Each game has 2 players. This loop produces 2 rows per game.
# Leaderboard ratings use elimination matching (known 0/1 key mismatch — see ISSUES)

def match_ratings(lb, player_nums):
    """Assign leaderboard ratings to player numbers by direct match then elimination."""
    rating_by_player = {}
    for pnum in player_nums:
        if pnum in lb:
            rating_by_player[pnum] = lb[pnum]
    assigned = set(rating_by_player.values())
    remaining = [v for k, v in lb.items() if v not in assigned]
    for pnum in player_nums:
        if pnum not in rating_by_player and remaining:
            rating_by_player[pnum] = remaining.pop(0)
    return rating_by_player


rows = []
for _, game in df.iterrows():
    player_nums  = sorted(game['de_players'].keys())
    age_up       = game['age_up_times']
    lb           = game['leaderboard_ratings']
    resign_pnum  = int(game['resign_player'])
    ratings      = match_ratings(lb, player_nums)

    for pnum in player_nums:
        dp         = game['de_players'][pnum]
        ages       = age_up.get(pnum, {})
        profile_id = dp.get('profile_id')

        rows.append({
            # ── Game identifiers ──
            'filepath':           game['filepath'],
            'player_num':         pnum,
            'profile_id':         profile_id,
            'player_name':        dp.get('name', b'').decode('utf-8', errors='replace') if isinstance(dp.get('name'), bytes) else str(dp.get('name', '')),
            'civilization_id':    dp.get('civilization_id'),

            # ── Constant filters (all games are Arabia/rated/1v1 by definition) ──
            'map_id':             9,
            'rated':              True,
            'num_players':        2,

            # ── Game-level fields ──
            'duration_min':       game['duration_min'],
            'result':             0 if pnum == resign_pnum else 1,  # 0=lost, 1=won

            # ── Age-up timings ──
            'feudal_time_min':    ages.get('feudal'),
            'castle_time_min':    ages.get('castle'),
            'imperial_time_min':  ages.get('imperial'),

            # ── Elo ──
            'elo':                ratings.get(pnum),

            # ── Identity flag ──
            'is_me':              profile_id == MY_PROFILE_ID,
        })

df_players = pd.DataFrame(rows)
print(f'Per-player DataFrame: {df_players.shape}  ({df_players.shape[0]} rows, {df_players.shape[1]} columns)')
print(f'  is_me rows:     {df_players["is_me"].sum()}  (should equal number of games = {len(df)})')
print(f'  opponent rows:  {(~df_players["is_me"]).sum()}')
df_players.head(4)

Per-player DataFrame: (350, 15)  (350 rows, 15 columns)
  is_me rows:     141  (should equal number of games = 175)
  opponent rows:  209


,filepath,player_num,profile_id,player_name,civilization_id,map_id,rated,num_players,duration_min,result,feudal_time_min,castle_time_min,imperial_time_min,elo,is_me
0,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\AgeIIDE_...,1,1332385,katzzlikemeowmix,10,9,True,2,22.06,1,6.68,20.16,NaN,1678.0,False
1,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\AgeIIDE_...,2,19535910,Bro Creation,4,9,True,2,22.06,0,6.73,NaN,NaN,1743.0,False
2,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\AgeIIDE_...,1,19535910,Bro Creation,29,9,True,2,27.02,0,7.41,19.48,NaN,1715.0,False
3,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\AgeIIDE_...,2,3134896,TheRealRuClEsHe,2,9,True,2,27.02,1,6.58,19.25,NaN,1665.0,True


In [9]:
# Quick sanity check on the flattened data
print('=== Null counts per column ===')
print(df_players.isnull().sum())
print()
print('=== Result distribution (should be ~50/50) ===')
print(df_players['result'].value_counts())
print()
print('=== is_me check ===')
me = df_players[df_players['is_me']]
print(f'  My rows: {len(me)}')
print(f'  My unique profile_ids: {me["profile_id"].nunique()}  (should be 1)')
print(f'  My player names: {me["player_name"].unique()}')

=== Null counts per column ===
filepath               0
player_num             0
profile_id             0
player_name            0
civilization_id        0
map_id                 0
rated                  0
num_players            0
duration_min           0
result                 0
feudal_time_min        2
castle_time_min       51
imperial_time_min    175
elo                    2
is_me                  0
dtype: int64

=== Result distribution (should be ~50/50) ===
result
1    175
0    175
Name: count, dtype: int64

=== is_me check ===
  My rows: 141
  My unique profile_ids: 1  (should be 1)
  My player names: ['TheRealRuClEsHe']
